# Stage 1: TrueSkill Ranking and Agreement Analysis

This notebook produces the final TrueSkill safety rankings for human crowdsourced votes and GPT-5.2 synthetic votes, and quantifies the agreement between the two sources at both the vote level and the ranking level.

**Pipeline:**
1. Install dependencies
2. Imports and configuration
3. Merge GPT vote files (25k + 100k + 100k = 225k combined)
4. Load and normalise all vote datasets
5. Behaviour-based filtering: human votes only (4 filters)
6. Filtering funnel diagnostic chart
7. TrueSkill rankings: human and GPT
8. Agreement Analysis A: vote-by-vote (25k GPT vs filtered human, same pairs)
9. Agreement Analysis B: ranking correlation (25k GPT ranking vs human ranking)
10. Sigma-stratified scatter plots (percentile rank)
11. Export final ranking CSVs for Stage 2

---

**Design note on GPT vote sets**

Three GPT vote files are used in this notebook:
- **25k votes** (`gpt52_votes_25k_20_06_26.csv`): collected on the same pairs as the human poll, validated through a prompt engineering evaluation using human votes as reference. Used for Agreement Analyses A and B.
- **100k votes #1** (`gpt52_votes_100k_20_06_26.csv`): additional pairs not in the 25k set, seed 99.
- **100k votes #2** (`gpt52_votes_100k_2_20_06_26.csv`): additional pairs not in the 25k set, seed 123.

All three files are merged into a single 225k GPT vote set for computing the final GPT TrueSkill ranking used in Stage 2. Agreement analyses use the 25k file only to ensure comparability with human votes on shared pairs.

## 1. Install dependencies

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'trueskill', '-q'], check=True)
print('Done.')

## 2. Imports and configuration


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import trueskill
from collections import Counter
from scipy.stats import spearmanr, kendalltau, pearsonr
from IPython.display import display
from sklearn.metrics import (
    confusion_matrix, classification_report,
    balanced_accuracy_score, accuracy_score,
    cohen_kappa_score, matthews_corrcoef,
    precision_recall_fscore_support,
)
warnings.filterwarnings('ignore', category=UserWarning)

# ============================================================
# PATHS
# ============================================================
HUMAN_PATH      = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Raw_data\human_votes_05_06_26.csv'
GPT_25K_PATH    = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_25k_20_06_26.csv'
GPT_100K_PATH_1 = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_100k_20_06_26.csv'
GPT_100K_PATH_2 = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\prompt_results\gpt52_votes_100k_2_20_06_26.csv'
OUTPUT_DIR      = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# HUMAN FILTERING PARAMETERS
# ============================================================
MIN_VOTES_SPEED    = 5     # minimum votes to apply speed filter
SPEED_THRESHOLD    = 2.0   # seconds: mean inter-vote time below this is flagged
MIN_VOTES_PATTERN  = 10    # minimum votes to apply streak and alternating filters
MAX_STREAK         = 10    # F3: flag if longest identical-vote run >= this
ALT_RATE_THRESHOLD = 0.90  # F4: flag if vote transition rate >= this

# ============================================================
# TRUESKILL
# Draw probabilities are computed automatically from the
# observed draw rate in each dataset. No manual setting needed.
# ============================================================

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
BLUE   = '#2171B5'
ORANGE = '#E8743B'
GREEN  = '#1B7837'

print('Configuration loaded.')
print(f'  Human votes      : {HUMAN_PATH}')
print(f'  GPT 25k          : {GPT_25K_PATH}')
print(f'  GPT 100k #1      : {GPT_100K_PATH_1}')
print(f'  GPT 100k #2      : {GPT_100K_PATH_2}')
print(f'  Output dir       : {OUTPUT_DIR}')

## 3. Merge GPT vote files

The three GPT vote files are concatenated into a single combined dataset of approximately 225k votes. This combined set is used to compute the final GPT TrueSkill ranking for Stage 2. The 25k file is also kept separately for the agreement analyses.

Column names are standardised across all three files: `model_id` is renamed to `user_id` where necessary, and image name paths are normalised to filename only.

In [ ]:
def normalise_image_name(path: str) -> str:
    return os.path.basename(str(path).strip())


def load_gpt_file(path, label):
    df = pd.read_csv(path)
    if 'model_id' in df.columns and 'user_id' not in df.columns:
        df = df.rename(columns={'model_id': 'user_id'})
    df = df.dropna(subset=['vote', 'image_name1', 'image_name2']).copy()
    df['image_name1'] = df['image_name1'].apply(normalise_image_name)
    df['image_name2'] = df['image_name2'].apply(normalise_image_name)
    df['vote']        = df['vote'].astype(int)
    df['source']      = label
    print(f'  {label:<12}: {len(df):>8,} rows  |  draw rate: {(df["vote"]==0).mean()*100:.2f}%')
    return df[['user_id', 'image_name1', 'image_name2', 'vote', 'source']].copy()


print('Loading GPT vote files...')
df_gpt_25k    = load_gpt_file(GPT_25K_PATH,    'gpt_25k')
df_gpt_100k_1 = load_gpt_file(GPT_100K_PATH_1, 'gpt_100k_1')
df_gpt_100k_2 = load_gpt_file(GPT_100K_PATH_2, 'gpt_100k_2')

# 225k combined: all three files (used for main Stage 2 ranking)
df_gpt_combined = pd.concat(
    [df_gpt_25k, df_gpt_100k_1, df_gpt_100k_2],
    ignore_index=True
)

# 200k only: excludes the validated 25k (used for robustness check in Stage 2)
df_gpt_200k = pd.concat(
    [df_gpt_100k_1, df_gpt_100k_2],
    ignore_index=True
)

def pair_key(df):
    return set(
        tuple(sorted([r.image_name1, r.image_name2]))
        for r in df.itertuples()
    )

pairs_25k    = pair_key(df_gpt_25k)
pairs_100k_1 = pair_key(df_gpt_100k_1)
pairs_100k_2 = pair_key(df_gpt_100k_2)

overlap_1 = len(pairs_25k & pairs_100k_1)
overlap_2 = len(pairs_25k & pairs_100k_2)
overlap_3 = len(pairs_100k_1 & pairs_100k_2)

print(f'\nCombined GPT dataset (225k):')
print(f'  Total rows          : {len(df_gpt_combined):>8,}')
print(f'  Draw rate           : {(df_gpt_combined["vote"]==0).mean()*100:.2f}%')
print(f'  Vote distribution   : {df_gpt_combined["vote"].value_counts().sort_index().to_dict()}')

print(f'\nGPT 200k only dataset:')
print(f'  Total rows          : {len(df_gpt_200k):>8,}')
print(f'  Draw rate           : {(df_gpt_200k["vote"]==0).mean()*100:.2f}%')

print(f'\nPair overlap between files:')
print(f'  25k vs 100k #1      : {overlap_1:,} shared pairs')
print(f'  25k vs 100k #2      : {overlap_2:,} shared pairs')
print(f'  100k #1 vs 100k #2  : {overlap_3:,} shared pairs')

## 4. Load and normalise human vote dataset

Human votes are loaded from the raw data file. Image names are normalised to filename only. Voting timestamps are parsed for use in the behaviour-based filtering step.

In [ ]:
print('Loading human votes...')
df_human_raw = pd.read_csv(HUMAN_PATH)
nan_before   = df_human_raw['vote'].isna().sum()
df_human_raw = df_human_raw.dropna(
    subset=['vote', 'image_name1', 'image_name2', 'voting_time']
).copy()
if nan_before:
    print(f'  Dropped {nan_before} rows with NaN values.')
df_human_raw['image_name1'] = df_human_raw['image_name1'].apply(normalise_image_name)
df_human_raw['image_name2'] = df_human_raw['image_name2'].apply(normalise_image_name)
df_human_raw['vote']        = df_human_raw['vote'].astype(int)
df_human_raw['voting_time'] = pd.to_datetime(
    df_human_raw['voting_time'], format='mixed', dayfirst=True
)
df_human = df_human_raw[[
    'user_id', 'image_name1', 'image_name2', 'vote', 'voting_time'
]].copy()

total_human_votes = len(df_human)
total_human_users = df_human['user_id'].nunique()

print(f'  Rows loaded         : {total_human_votes:,}')
print(f'  Unique users        : {total_human_users:,}')
print(f'  Vote distribution   : {df_human["vote"].value_counts().sort_index().to_dict()}')
print(f'  Draw rate           : {(df_human["vote"]==0).mean()*100:.2f}%')

# pair overlap: human vs 25k GPT (agreement analysis will use these)
human_pairs = pair_key(df_human)
overlap_h_g = len(human_pairs & pairs_25k)
print(f'\nPair overlap — human vs GPT 25k : {overlap_h_g:,}')
if overlap_h_g < 100:
    print('  WARNING: very few overlapping pairs. Agreement Analysis A will not be meaningful.')

## 5. Behaviour-based filtering: human votes only

Four sequential filters are applied to the human vote dataset to remove low-quality responses. GPT votes are not filtered.

- **F1 Directional bias**: max vote-option proportion exceeds a dynamic threshold that tightens for users with more votes
- **F2 Speed**: mean inter-vote time below 2 seconds, applied to users with at least 5 votes
- **F3 Consecutive streak**: longest identical-vote run >= 10, applied to users with at least 10 votes
- **F4 Alternating pattern**: vote transition rate >= 0.90, applied to users with at least 10 votes

In [ ]:
def compute_user_stats(df):
    counts   = df.groupby('user_id').size().rename('vote_count')
    vote_props = (
        df.groupby('user_id')['vote']
          .value_counts(normalize=True)
          .rename('prop').reset_index()
    )
    max_prop = vote_props.groupby('user_id')['prop'].max().rename('max_option_prop')
    df_sorted = df.sort_values(['user_id', 'voting_time']).copy()
    dt = df_sorted.groupby('user_id')['voting_time'].diff().dt.total_seconds()
    df_sorted['dt'] = dt
    mean_dt = (
        df_sorted.groupby('user_id')['dt']
        .mean().replace({np.nan: np.inf})
        .rename('mean_dt_seconds')
    )
    return pd.concat([counts, max_prop, mean_dt], axis=1)


def compute_bias_threshold(vote_count):
    if vote_count > 100:  return 0.70
    elif vote_count > 50: return 0.75
    elif vote_count > 10: return 0.80
    else:                 return np.nan


def compute_pattern_stats(df):
    df_sorted = df.sort_values(['user_id', 'voting_time']).copy()
    results = []
    for user_id, grp in df_sorted.groupby('user_id'):
        votes = grp['vote'].tolist()
        n = len(votes)
        max_streak = current_streak = 1
        for i in range(1, n):
            if votes[i] == votes[i-1]:
                current_streak += 1
                max_streak = max(max_streak, current_streak)
            else:
                current_streak = 1
        transition_rate = (
            sum(votes[i] != votes[i-1] for i in range(1, n)) / (n-1)
            if n >= 2 else np.nan
        )
        results.append({'user_id': user_id, 'max_streak': max_streak,
                        'transition_rate': transition_rate})
    return pd.DataFrame(results).set_index('user_id')


print('Computing per-user statistics...')
user_stats_before = compute_user_stats(df_human)
user_stats_before['bias_threshold'] = user_stats_before['vote_count'].apply(
    compute_bias_threshold
)
print(f'Raw data: {total_human_votes:,} votes from {total_human_users:,} users')

# F1: Directional bias
bias_applicable = ~user_stats_before['bias_threshold'].isna()
bias_bad_users  = user_stats_before.index[
    bias_applicable &
    (user_stats_before['max_option_prop'] >= user_stats_before['bias_threshold'])
].tolist()
df_after_f1    = df_human[~df_human['user_id'].isin(bias_bad_users)].copy()
votes_after_f1 = len(df_after_f1)
users_after_f1 = df_after_f1['user_id'].nunique()
print(f'\nF1 Directional bias:')
print(f'  Users eligible : {bias_applicable.sum()} / {len(user_stats_before)}')
print(f'  Users removed  : {len(bias_bad_users)}')
print(f'  Votes remaining: {votes_after_f1:,} (removed {total_human_votes - votes_after_f1:,})')

# F2: Speed
user_stats_f1   = compute_user_stats(df_after_f1)
speed_bad_users = user_stats_f1.index[
    (user_stats_f1['vote_count'] >= MIN_VOTES_SPEED) &
    (user_stats_f1['mean_dt_seconds'] < SPEED_THRESHOLD)
].tolist()
df_after_f2    = df_after_f1[~df_after_f1['user_id'].isin(speed_bad_users)].copy()
votes_after_f2 = len(df_after_f2)
users_after_f2 = df_after_f2['user_id'].nunique()
print(f'\nF2 Speed (mean inter-vote < {SPEED_THRESHOLD}s):')
print(f'  Users removed  : {len(speed_bad_users)}')
print(f'  Votes remaining: {votes_after_f2:,} (removed {votes_after_f1 - votes_after_f2:,})')

# F3: Consecutive streak
user_stats_f2    = compute_user_stats(df_after_f2)
pattern_stats    = compute_pattern_stats(df_after_f2)
user_stats_f2    = user_stats_f2.join(pattern_stats)
eligible_mask    = user_stats_f2['vote_count'] >= MIN_VOTES_PATTERN
streak_bad_users = user_stats_f2.index[
    eligible_mask & (user_stats_f2['max_streak'] >= MAX_STREAK)
].tolist()
df_after_f3    = df_after_f2[~df_after_f2['user_id'].isin(streak_bad_users)].copy()
votes_after_f3 = len(df_after_f3)
users_after_f3 = df_after_f3['user_id'].nunique()
print(f'\nF3 Consecutive streak (max run >= {MAX_STREAK}):')
print(f'  Users eligible : {eligible_mask.sum()} / {len(user_stats_f2)}')
print(f'  Users removed  : {len(streak_bad_users)}')
print(f'  Votes remaining: {votes_after_f3:,} (removed {votes_after_f2 - votes_after_f3:,})')

# F4: Alternating pattern
user_stats_f3    = compute_user_stats(df_after_f3)
pattern_stats_f3 = compute_pattern_stats(df_after_f3)
user_stats_f3    = user_stats_f3.join(pattern_stats_f3)
eligible_mask_f3 = user_stats_f3['vote_count'] >= MIN_VOTES_PATTERN
alt_bad_users    = user_stats_f3.index[
    eligible_mask_f3 & (user_stats_f3['transition_rate'] >= ALT_RATE_THRESHOLD)
].tolist()
df_after_f4    = df_after_f3[~df_after_f3['user_id'].isin(alt_bad_users)].copy()
votes_after_f4 = len(df_after_f4)
users_after_f4 = df_after_f4['user_id'].nunique()
print(f'\nF4 Alternating pattern (transition rate >= {ALT_RATE_THRESHOLD}):')
print(f'  Users eligible : {eligible_mask_f3.sum()} / {len(user_stats_f3)}')
print(f'  Users removed  : {len(alt_bad_users)}')
print(f'  Votes remaining: {votes_after_f4:,} (removed {votes_after_f3 - votes_after_f4:,})')

filtered_human_df   = df_after_f4.copy()
final_votes         = votes_after_f4
final_users         = users_after_f4
total_removed       = total_human_votes - final_votes
unreliable_user_ids = sorted(
    set(bias_bad_users) | set(speed_bad_users) |
    set(streak_bad_users) | set(alt_bad_users)
)
print(f'\nFiltering complete.')
print(f'  Final votes          : {final_votes:,}')
print(f'  Final users          : {final_users:,}')
print(f'  Total votes removed  : {total_removed:,} ({total_removed/total_human_votes*100:.1f}%)')
print(f'  Unreliable users     : {len(unreliable_user_ids)}')

## 6. Filtering funnel diagnostic

A mosaic chart showing the before and after distribution for each filter.

In [ ]:
import matplotlib.patches as mpatches

s0 = compute_user_stats(df_human).join(compute_pattern_stats(df_human), how='left')
s1 = compute_user_stats(df_after_f1).join(compute_pattern_stats(df_after_f1), how='left')
s2 = compute_user_stats(df_after_f2).join(compute_pattern_stats(df_after_f2), how='left')
s3 = compute_user_stats(df_after_f3).join(compute_pattern_stats(df_after_f3), how='left')
s4 = compute_user_stats(df_after_f4).join(compute_pattern_stats(df_after_f4), how='left')

filters = [
    ('F1 Directional bias',   'max_option_prop',  'Max option proportion',          0.80, 'above', 1.0,  25, s0, s1),
    ('F2 Speed',              'mean_dt_seconds',  'Mean inter-vote time (s)',         2.0,  'below', 60.0, 35, s1, s2),
    ('F3 Consecutive streak', 'max_streak',       'Longest identical streak (votes)', 10.0, 'above', 40.0, 30, s2, s3),
    ('F4 Alternating pattern','transition_rate',  'Vote transition rate',             0.90, 'above', 1.0,  25, s3, s4),
]

fig, axes = plt.subplots(4, 2, figsize=(12, 13))
fig.patch.set_facecolor('white')
axes[0, 0].set_title('Before filter', fontsize=11, fontweight='bold', pad=10)
axes[0, 1].set_title('After filter',  fontsize=11, fontweight='bold', pad=10)

for row_idx, (label, col, xlabel, threshold, direction,
              cap, n_bins, before_df, after_df) in enumerate(filters):
    for col_idx, (stats_df, is_before) in enumerate([
        (before_df, True), (after_df, False)
    ]):
        ax = axes[row_idx, col_idx]
        ax.set_facecolor('white')
        vals = stats_df[col].replace([np.inf, -np.inf], np.nan).dropna()
        vals = vals[vals <= cap]
        if len(vals) == 0:
            ax.axis('off')
            continue
        removed_mask = pd.Series(False, index=vals.index)
        if is_before:
            removed_mask = vals >= threshold if direction == 'above' else vals < threshold
        bins = np.linspace(0, cap, n_bins)
        ax.hist(vals[~removed_mask], bins=bins, color=BLUE, alpha=0.85,
                edgecolor='white', linewidth=0.4, zorder=3)
        if is_before and removed_mask.sum() > 0:
            ax.hist(vals[removed_mask], bins=bins, color='#e74c3c', alpha=0.75,
                    edgecolor='white', linewidth=0.4, zorder=3)
        ax.axvline(threshold, color='#c0392b', linewidth=1.6, linestyle='--', zorder=4)
        ax.text(0.98, 0.97, f'n = {len(vals):,}',
                transform=ax.transAxes, ha='right', va='top', fontsize=8, color='#555')
        ax.set_xlabel(xlabel, fontsize=8.5)
        ax.set_ylabel('Users', fontsize=8.5)
        ax.tick_params(labelsize=8)
        ax.spines[['top', 'right']].set_visible(False)
        ax.spines['left'].set_color('#cccccc')
        ax.spines['bottom'].set_color('#cccccc')
        ax.grid(axis='y', alpha=0.2, linestyle='--')
    axes[row_idx, 0].annotate(
        label, xy=(-0.26, 0.5), xycoords='axes fraction',
        fontsize=9, fontweight='bold', color='#222',
        rotation=90, ha='center', va='center'
    )

kept_patch    = mpatches.Patch(color=BLUE,      alpha=0.85, label='Retained users')
removed_patch = mpatches.Patch(color='#e74c3c', alpha=0.75, label='Removed users')
fig.legend(
    handles=[kept_patch, removed_patch],
    loc='lower center', ncol=2, fontsize=9.5,
    frameon=True, framealpha=0.95,
    bbox_to_anchor=(0.5, -0.01), edgecolor='#cccccc'
)
fig.text(0.5, -0.025, 'Dashed line = exclusion threshold',
         ha='center', fontsize=9, color='#c0392b', style='italic')
fig.suptitle(
    'Behaviour-based vote filtering: before and after each filter',
    fontsize=12, fontweight='bold', y=1.01
)
plt.tight_layout(h_pad=2.5, w_pad=2.0)
plt.show()

## 7. TrueSkill rankings

TrueSkill is run separately for human and GPT votes. Draw probabilities are computed from the observed draw rate in each dataset.

Two GPT rankings are produced:
- **GPT 25k ranking**: based on the validated 25k votes only. Used for Agreement Analyses A and B.
- **GPT 225k ranking**: based on all three vote files combined. Used as the final GPT ranking for Stage 2.

The TrueSkill score is defined as `mu - 3 * sigma`, which is a conservative lower bound on the true skill estimate.

In [ ]:
def run_trueskill(vote_df, draw_prob, label):
    env     = trueskill.TrueSkill(draw_probability=draw_prob)
    ratings = {}

    for _, row in vote_df.iterrows():
        img1 = row['image_name1']
        img2 = row['image_name2']
        vote = row['vote']
        if img1 not in ratings: ratings[img1] = env.create_rating()
        if img2 not in ratings: ratings[img2] = env.create_rating()
        r1, r2 = ratings[img1], ratings[img2]
        if vote == 1:
            ratings[img1], ratings[img2] = env.rate_1vs1(r1, r2)
        elif vote == 2:
            ratings[img2], ratings[img1] = env.rate_1vs1(r2, r1)
        else:
            ratings[img1], ratings[img2] = env.rate_1vs1(r1, r2, drawn=True)

    records = [
        {'image_name': img, 'mu': r.mu, 'sigma': r.sigma,
         'ts_score': r.mu - 3 * r.sigma}
        for img, r in ratings.items()
    ]
    df_out = pd.DataFrame(records).sort_values('ts_score', ascending=False).reset_index(drop=True)
    df_out['rank'] = df_out['ts_score'].rank(ascending=False, method='min').astype(int)

    print(f'  {label}:')
    print(f'    Images ranked  : {len(df_out):,}')
    print(f'    Score range    : [{df_out["ts_score"].min():.4f}, {df_out["ts_score"].max():.4f}]')
    print(f'    Sigma range    : [{df_out["sigma"].min():.4f}, {df_out["sigma"].max():.4f}]')
    return df_out


print('Computing TrueSkill rankings...')

# human
draw_prob_human = (filtered_human_df['vote'] == 0).mean()
print(f'\nHuman draw probability (filtered): {draw_prob_human:.4f}')
human_rank_df = run_trueskill(filtered_human_df, draw_prob_human, 'Human')
human_rank_df.columns = ['image_name', 'human_mu', 'human_sigma', 'human_ts_score', 'rank_human']

# GPT 25k
draw_prob_gpt_25k = (df_gpt_25k['vote'] == 0).mean()
print(f'\nGPT 25k draw probability: {draw_prob_gpt_25k:.4f}')
gpt_rank_25k_df = run_trueskill(df_gpt_25k, draw_prob_gpt_25k, 'GPT 25k')
gpt_rank_25k_df.columns = ['image_name', 'gpt_mu', 'gpt_sigma', 'gpt_ts_score', 'rank_gpt']

# GPT 225k combined
draw_prob_gpt_combined = (df_gpt_combined['vote'] == 0).mean()
print(f'\nGPT 225k draw probability: {draw_prob_gpt_combined:.4f}')
gpt_rank_combined_df = run_trueskill(df_gpt_combined, draw_prob_gpt_combined, 'GPT 225k combined')
gpt_rank_combined_df.columns = ['image_name', 'gpt_mu', 'gpt_sigma', 'gpt_ts_score', 'rank_gpt']

# GPT 200k only (excludes validated 25k — robustness check for Stage 2)
draw_prob_gpt_200k = (df_gpt_200k['vote'] == 0).mean()
print(f'\nGPT 200k draw probability: {draw_prob_gpt_200k:.4f}')
gpt_rank_200k_df = run_trueskill(df_gpt_200k, draw_prob_gpt_200k, 'GPT 200k only')
gpt_rank_200k_df.columns = ['image_name', 'gpt_mu', 'gpt_sigma', 'gpt_ts_score', 'rank_gpt']

print('\nAll rankings complete.')

## 8. Agreement Analysis A: vote-by-vote

Vote-by-vote comparison between filtered human votes and GPT 25k votes on the same pairs. The human majority vote is used as the reference label. GPT voted once per pair so the single GPT vote is used directly.

Metrics: accuracy, balanced accuracy, Cohen kappa, Matthews correlation coefficient, per-class precision, recall and F1.

In [ ]:
# find overlapping pairs
human_pairs_filtered = set(
    tuple(sorted([r.image_name1, r.image_name2]))
    for r in filtered_human_df.itertuples()
)
gpt_25k_pairs = set(
    tuple(sorted([r.image_name1, r.image_name2]))
    for r in df_gpt_25k.itertuples()
)
overlapping_pairs = human_pairs_filtered & gpt_25k_pairs

print(f'Unique pairs — human (filtered) : {len(human_pairs_filtered):,}')
print(f'Unique pairs — GPT 25k          : {len(gpt_25k_pairs):,}')
print(f'Overlapping pairs               : {len(overlapping_pairs):,}')

if len(overlapping_pairs) < 100:
    print('WARNING: too few overlapping pairs. Agreement Analysis A will not be meaningful.')
else:
    # build pair key column for merging
    def add_pair_key(df):
        df = df.copy()
        df['pair_key'] = df.apply(
            lambda r: tuple(sorted([r['image_name1'], r['image_name2']])), axis=1
        )
        return df

    human_keyed = add_pair_key(filtered_human_df)
    gpt_keyed   = add_pair_key(df_gpt_25k)

    # human majority vote per pair
    human_majority = (
        human_keyed.groupby('pair_key')['vote']
        .agg(lambda x: x.value_counts().index[0])
        .reset_index().rename(columns={'vote': 'human_vote'})
    )
    gpt_single = gpt_keyed[['pair_key', 'vote']].rename(columns={'vote': 'gpt_vote'})

    overlap_df = human_majority.merge(gpt_single, on='pair_key', how='inner')
    print(f'\nOverlapping pairs after merge: {len(overlap_df):,}')

    y_true = overlap_df['human_vote'].astype(int).values
    y_pred = overlap_df['gpt_vote'].astype(int).values
    vote_labels = [0, 1, 2]

    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    kappa   = cohen_kappa_score(y_true, y_pred)
    mcc     = matthews_corrcoef(y_true, y_pred)
    _, _, f1_macro,  _ = precision_recall_fscore_support(
        y_true, y_pred, labels=vote_labels, average='macro',    zero_division=0)
    _, _, f1_weight, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=vote_labels, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=vote_labels)

    print(f'\n===== AGREEMENT ANALYSIS A: Vote-by-vote =====')
    print(f'  n overlapping pairs    : {len(overlap_df):,}')
    print(f'  Accuracy               : {acc:.4f}')
    print(f'  Balanced accuracy      : {bal_acc:.4f}')
    print(f'  Cohen kappa            : {kappa:.4f}')
    print(f'  Matthews corrcoef      : {mcc:.4f}')
    print(f'  Macro F1               : {f1_macro:.4f}')
    print(f'  Weighted F1            : {f1_weight:.4f}')
    print(f'\n  Classification report:')
    display(pd.DataFrame(
        classification_report(
            y_true, y_pred,
            labels=vote_labels,
            target_names=['Equal (0)', 'Img A safer (1)', 'Img B safer (2)'],
            zero_division=0, output_dict=True
        )
    ).T)

    # draw class analysis
    decisive_mask = y_true != 0
    n_decisive    = decisive_mask.sum()
    n_draw_human  = (y_true == 0).sum()
    acc_decisive  = (y_true[decisive_mask] == y_pred[decisive_mask]).mean()
    draw_recall   = (y_pred[y_true == 0] == 0).mean() if n_draw_human > 0 else 0.0
    gpt_draw_rate_a = (y_pred == 0).mean()

    print(f'\n  Draw class analysis:')
    print(f'    Human draw pairs              : {n_draw_human:,} ({n_draw_human/len(y_true)*100:.1f}%)')
    print(f'    GPT draw rate                 : {gpt_draw_rate_a*100:.2f}%')
    print(f'    Draw recall (GPT on human draws): {draw_recall*100:.2f}%')
    print(f'\n  Agreement on decisive pairs (human voted 1 or 2):')
    print(f'    n pairs    : {n_decisive:,} ({n_decisive/len(y_true)*100:.1f}% of total)')
    print(f'    Agreement  : {acc_decisive*100:.2f}%')
    print(f'    Baseline   : 50.00% (random between two classes)')
    print(f'    Lift       : +{(acc_decisive - 0.5)*100:.2f}pp')

## 9. Agreement Analysis B: ranking correlation

Ranking correlation between the human TrueSkill ranking and the GPT 25k TrueSkill ranking on images present in both. Spearman rho, Kendall tau and Pearson r are reported. Jaccard index is computed for the top K images by rank.

In [ ]:
merged = human_rank_df.merge(gpt_rank_25k_df, on='image_name', how='inner')
n      = len(merged)

print(f'Images in human ranking  : {len(human_rank_df):,}')
print(f'Images in GPT 25k ranking: {len(gpt_rank_25k_df):,}')
print(f'Overlapping images       : {n:,}')

rho_score, p_rho   = spearmanr(merged['human_ts_score'], merged['gpt_ts_score'])
rho_rank,  p_rho_r = spearmanr(merged['rank_human'],     merged['rank_gpt'])
tau,       p_tau   = kendalltau(merged['rank_human'],    merged['rank_gpt'])
r_pears,   p_r     = pearsonr(merged['human_ts_score'],  merged['gpt_ts_score'])

print(f'\n===== AGREEMENT ANALYSIS B: Ranking correlation =====')
print(f'  Spearman rho (scores) : {rho_score:.4f}  (p={p_rho:.3e})')
print(f'  Spearman rho (ranks)  : {rho_rank:.4f}  (p={p_rho_r:.3e})')
print(f'  Kendall tau  (ranks)  : {tau:.4f}  (p={p_tau:.3e})')
print(f'  Pearson r    (scores) : {r_pears:.4f}  (p={p_r:.3e})')

print(f'\n  Jaccard index (top-K):')
for k in [10, 20, 30, 100, 500]:
    top_h  = set(merged[merged['rank_human'] <= k]['image_name'])
    top_g  = set(merged[merged['rank_gpt']   <= k]['image_name'])
    shared = len(top_h & top_g)
    union  = len(top_h | top_g)
    j      = shared / union if union > 0 else np.nan
    print(f'    Top {k:>4}  ->  Jaccard = {j:.4f}  ({shared} shared out of {union} in union)')

## 10. Sigma-stratified scatter plots

Six panels showing the correlation between human and GPT 25k safety score percentile ranks at increasing levels of certainty. Each panel restricts the comparison to images where the maximum sigma across both sources is below the stated quantile threshold.

Percentile rank is used instead of raw TrueSkill scores to place both sources on a common 0 to 100 scale, making the 1:1 reference line meaningful and avoiding the bimodal gap caused by TrueSkill's draw mechanism.

In [ ]:
# ── PERCENTILE RANK VERSION ───────────────────────────────────────────────────
merged['max_sigma'] = merged[['human_sigma', 'gpt_sigma']].max(axis=1)
merged['human_pct'] = merged['human_ts_score'].rank(pct=True) * 100
merged['gpt_pct']   = merged['gpt_ts_score'].rank(pct=True) * 100

quantiles    = [0.01, 0.05, 0.10, 0.25, 0.50, 0.95]
thresholds   = merged['max_sigma'].quantile(quantiles).values
panel_labels = ['Top 1%\n(most certain)', 'Top 5%', 'Top 10%',
                'Top 25%', 'Top 50%', 'Top 95%\n(all images)']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

for idx, (q, thr, panel_label) in enumerate(
    zip(quantiles, thresholds, panel_labels)
):
    ax     = axes_flat[idx]
    ax.set_facecolor('white')
    subset = merged[merged['max_sigma'] <= thr].copy()
    n_sub  = len(subset)

    if n_sub < 3:
        ax.axis('off')
        continue

    x = subset['human_pct'].values
    y = subset['gpt_pct'].values

    certainty      = 1.0 / np.maximum(subset['max_sigma'].values, 1e-6)
    certainty_norm = (certainty - certainty.min()) / (
        certainty.max() - certainty.min() + 1e-12
    )
    sizes = 5 + certainty_norm * 20

    ax.scatter(x, y, s=sizes, alpha=0.4,
               color=BLUE, linewidths=0, rasterized=True, zorder=3)

    ax.plot([0, 100], [0, 100],
            color='#c0392b', linewidth=1.4,
            linestyle='--', alpha=0.8, zorder=4, label='1:1 line')

    w   = certainty
    W   = w.sum()
    xw  = (w * x).sum() / W
    yw  = (w * y).sum() / W
    den = (w * (x - xw) ** 2).sum()
    if den > 0:
        b1  = (w * (x - xw) * (y - yw)).sum() / den
        b0  = yw - b1 * xw
        x_r = np.linspace(0, 100, 100)
        ax.plot(x_r, b1 * x_r + b0,
                color='#e67e22', linewidth=1.6,
                linestyle='-', alpha=0.9, zorder=5, label='Weighted fit')

    rho, _ = spearmanr(x, y)
    r, _   = pearsonr(x, y)

    ax.text(0.97, 0.05,
            f'Spearman rho = {rho:.3f}\nPearson r = {r:.3f}\nn = {n_sub:,}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8,
            color='#333333',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#dddddd', alpha=0.9))

    ax.set_title(f'{panel_label}  (max sigma <= {thr:.2f})',
                 fontsize=9, fontweight='bold', pad=6)
    ax.set_xlabel('Human percentile rank', fontsize=8.5)
    ax.set_ylabel('GPT percentile rank', fontsize=8.5)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.tick_params(labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_color('#cccccc')
    ax.spines['bottom'].set_color('#cccccc')
    ax.grid(alpha=0.2, linestyle='--', color='#aaaaaa')

    if idx == 0:
        ax.legend(fontsize=7.5, frameon=True, framealpha=0.9,
                  loc='upper left', edgecolor='#cccccc')

fig.suptitle(
    'Sigma-stratified scatter plots: GPT vs Human safety score percentile rank\n'
    'Each panel includes images where max(human sigma, GPT sigma) is below the stated threshold\n'
    'Dashed red line = 1:1 agreement  |  Orange line = weighted regression  |  '
    'Dot size proportional to certainty',
    fontsize=10, fontweight='bold', y=1.02, color='#111111'
)
plt.tight_layout(h_pad=2.5, w_pad=2.0)
plt.show()

## Second chart for Luis

In [ ]:
# ── NORMALISED MU VERSION — scaling applied per subset ────────────────────────
from sklearn.preprocessing import MinMaxScaler

merged['max_sigma'] = merged[['human_sigma', 'gpt_sigma']].max(axis=1)

quantiles    = [0.01, 0.05, 0.10, 0.25, 0.50, 0.95]
thresholds   = merged['max_sigma'].quantile(quantiles).values
panel_labels = ['Top 1%\n(most certain)', 'Top 5%', 'Top 10%',
                'Top 25%', 'Top 50%', 'Top 95%\n(all images)']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.patch.set_facecolor('white')
axes_flat = axes.flatten()

for idx, (q, thr, panel_label) in enumerate(
    zip(quantiles, thresholds, panel_labels)
):
    ax     = axes_flat[idx]
    ax.set_facecolor('white')
    subset = merged[merged['max_sigma'] <= thr].copy()
    n_sub  = len(subset)

    if n_sub < 3:
        ax.axis('off')
        continue

    # scale independently within each subset
    scaler_h = MinMaxScaler()
    scaler_g = MinMaxScaler()
    x = scaler_h.fit_transform(subset[['human_ts_score']]).flatten()
    y = scaler_g.fit_transform(subset[['gpt_ts_score']]).flatten()

    certainty      = 1.0 / np.maximum(subset['max_sigma'].values, 1e-6)
    certainty_norm = (certainty - certainty.min()) / (
        certainty.max() - certainty.min() + 1e-12
    )
    sizes = 5 + certainty_norm * 20

    ax.scatter(x, y, s=sizes, alpha=0.4,
               color=BLUE, linewidths=0, rasterized=True, zorder=3)

    ax.plot([0, 1], [0, 1],
            color='#c0392b', linewidth=1.4,
            linestyle='--', alpha=0.8, zorder=4, label='1:1 line')

    w   = certainty
    W   = w.sum()
    xw  = (w * x).sum() / W
    yw  = (w * y).sum() / W
    den = (w * (x - xw) ** 2).sum()
    if den > 0:
        b1  = (w * (x - xw) * (y - yw)).sum() / den
        b0  = yw - b1 * xw
        x_r = np.linspace(0, 1, 100)
        ax.plot(x_r, b1 * x_r + b0,
                color='#e67e22', linewidth=1.6,
                linestyle='-', alpha=0.9, zorder=5, label='Weighted fit')

    rho, _ = spearmanr(x, y)
    r, _   = pearsonr(x, y)

    ax.text(0.97, 0.05,
            f'Spearman rho = {rho:.3f}\nPearson r = {r:.3f}\nn = {n_sub:,}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8,
            color='#333333',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#dddddd', alpha=0.9))

    ax.set_title(f'{panel_label}  (max sigma <= {thr:.2f})',
                 fontsize=9, fontweight='bold', pad=6)
    ax.set_xlabel('Human normalised safety score', fontsize=8.5)
    ax.set_ylabel('GPT normalised safety score', fontsize=8.5)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f'{v:.1f}')
    )
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f'{v:.1f}')
    )
    ax.tick_params(labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_color('#cccccc')
    ax.spines['bottom'].set_color('#cccccc')
    ax.grid(alpha=0.2, linestyle='--', color='#aaaaaa')

    if idx == 0:
        ax.legend(fontsize=7.5, frameon=True, framealpha=0.9,
                  loc='upper left', edgecolor='#cccccc')

fig.suptitle(
    'Sigma-stratified scatter plots: GPT vs Human normalised TrueSkill safety score\n'
    'Each panel includes images where max(human sigma, GPT sigma) is below the stated threshold\n'
    'Normalised mu scaled independently per subset  |  Spearman rho is rank-based and identical across both versions\n'
    'Dashed red line = 1:1 agreement  |  Orange line = weighted regression  |  '
    'Dot size proportional to certainty',
    fontsize=10, fontweight='bold', y=1.02, color='#111111'
)
plt.tight_layout(h_pad=2.5, w_pad=2.0)
plt.show()

In [ ]:
# ── DIAGNOSTIC: verify correlations differ between the two versions ───────────
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import spearmanr, pearsonr
import numpy as np

merged['max_sigma'] = merged[['human_sigma', 'gpt_sigma']].max(axis=1)
merged['human_pct'] = merged['human_ts_score'].rank(pct=True) * 100
merged['gpt_pct']   = merged['gpt_ts_score'].rank(pct=True) * 100

quantiles  = [0.01, 0.05, 0.10, 0.25, 0.50, 0.95]
thresholds = merged['max_sigma'].quantile(quantiles).values

print(f'{"Panel":<12} {"n":>6} {"Pct_rho":>10} {"Pct_r":>10} {"Mu_rho":>10} {"Mu_r":>10} {"rho_same":>10} {"r_same":>10}')
print('-' * 80)

for q, thr in zip(quantiles, thresholds):
    subset = merged[merged['max_sigma'] <= thr].copy()
    n_sub  = len(subset)
    if n_sub < 3:
        continue

    # percentile rank version
    x_pct = subset['human_pct'].values
    y_pct = subset['gpt_pct'].values
    rho_pct, _ = spearmanr(x_pct, y_pct)
    r_pct, _   = pearsonr(x_pct, y_pct)

    # normalised mu version — scaled per subset
    scaler_h = MinMaxScaler()
    scaler_g = MinMaxScaler()
    x_mu = scaler_h.fit_transform(subset[['human_ts_score']]).flatten()
    y_mu = scaler_g.fit_transform(subset[['gpt_ts_score']]).flatten()
    rho_mu, _ = spearmanr(x_mu, y_mu)
    r_mu, _   = pearsonr(x_mu, y_mu)

    rho_same = 'YES' if abs(rho_pct - rho_mu) < 0.0001 else 'NO'
    r_same   = 'YES' if abs(r_pct - r_mu) < 0.0001 else 'NO'

    print(f'{q*100:.0f}% subset {n_sub:>6,} {rho_pct:>10.4f} {r_pct:>10.4f} {rho_mu:>10.4f} {r_mu:>10.4f} {rho_same:>10} {r_same:>10}')

## 11. Export final ranking CSVs

Two CSV files are saved for use in Stage 2:
- **Human ranking**: based on 25k filtered votes
- **GPT ranking**: based on all 225k combined votes

Both files use stable fixed filenames so Stage 2 does not need updating when Stage 1 is rerun.

In [ ]:
def save_ranking(df, label, draw_prob, output_dir, filename):
    path = os.path.join(output_dir, filename)
    df.to_csv(path, index=False)
    mu_col    = [c for c in df.columns if 'mu'    in c and 'sigma' not in c][0]
    sigma_col = [c for c in df.columns if 'sigma' in c][0]
    score_col = [c for c in df.columns if 'ts_score' in c][0]
    print(f'{label} ranking saved: {path}')
    print(f'  Rows        : {len(df):,}')
    print(f'  Draw prob   : {draw_prob:.4f}')
    print(f'  Score range : [{df[score_col].min():.4f}, {df[score_col].max():.4f}]')
    print(f'  Sigma range : [{df[sigma_col].min():.4f}, {df[sigma_col].max():.4f}]')
    display(df.head(3))


save_ranking(
    human_rank_df,
    label      = 'Human',
    draw_prob  = draw_prob_human,
    output_dir = OUTPUT_DIR,
    filename   = 'human_trueskill_rankings_final_20_06_26.csv'
)

save_ranking(
    gpt_rank_combined_df,
    label      = 'GPT (225k combined)',
    draw_prob  = draw_prob_gpt_combined,
    output_dir = OUTPUT_DIR,
    filename   = 'gpt_trueskill_rankings_final_20_06_26.csv'
)

save_ranking(
    gpt_rank_200k_df,
    label      = 'GPT (200k only)',
    draw_prob  = draw_prob_gpt_200k,
    output_dir = OUTPUT_DIR,
    filename   = 'gpt_trueskill_rankings_200k_20_06_26.csv'
)

print('\n===== Stage 1 complete. Three ranking CSVs saved for Stage 2. =====')
print(f'  human_trueskill_rankings_final_20_06_26.csv')
print(f'  gpt_trueskill_rankings_final_20_06_26.csv')
print(f'  gpt_trueskill_rankings_200k_20_06_26.csv')

## Charts for ppaper

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

FONT_FAMILY = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY

HUMAN_RAW_COLOR      = '#1B7837'
HUMAN_FILTERED_COLOR = '#52A860'
GPT_COLOR            = '#2171B5'
OUTPUT_DIR = r'C:\Users\researcher\Documents\2. Job\1. tasks\14. attempt\Ranking_data'

# ── compute distributions ─────────────────────────────────────────────────────
hr_total  = len(df_human)
hr_counts = df_human['vote'].value_counts().sort_index()
hr_pcts   = (hr_counts / hr_total * 100).reindex([0, 1, 2], fill_value=0)

hf_total  = len(filtered_human_df)
hf_counts = filtered_human_df['vote'].value_counts().sort_index()
hf_pcts   = (hf_counts / hf_total * 100).reindex([0, 1, 2], fill_value=0)

g_total  = len(df_gpt_25k)
g_counts = df_gpt_25k['vote'].value_counts().sort_index()
g_pcts   = (g_counts / g_total * 100).reindex([0, 1, 2], fill_value=0)

labels = ['Draw\n(no clear difference)', 'Image A safer', 'Image B safer']
x      = np.arange(len(labels))
width  = 0.25

# ── rounding adjustment to force sum to 100.0% ───────────────────────────────
def adjust_to_100(pcts, decimals=1):
    rounded = [round(p, decimals) for p in pcts]
    diff    = round(100.0 - sum(rounded), decimals)
    if diff != 0:
        idx          = rounded.index(max(rounded))
        rounded[idx] = round(rounded[idx] + diff, decimals)
    return rounded

# ── figure ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars_hr = ax.bar(x - width,     hr_pcts.values, width,
                 color=HUMAN_RAW_COLOR, alpha=0.5,
                 edgecolor='white', label=f'Human raw (n={hr_total:,})')
bars_hf = ax.bar(x,             hf_pcts.values, width,
                 color=HUMAN_RAW_COLOR, alpha=0.9,
                 edgecolor='white', label=f'Human filtered (n={hf_total:,})')
bars_g  = ax.bar(x + width,     g_pcts.values,  width,
                 color=GPT_COLOR, alpha=0.9,
                 edgecolor='white', label=f'GPT-5.2 validated (n={g_total:,})')

y_max = max(hr_pcts.max(), hf_pcts.max(), g_pcts.max())
ax.set_ylim(0, y_max * 1.35)

for bars, pcts in [(bars_hr, hr_pcts.values),
                   (bars_hf, hf_pcts.values),
                   (bars_g,  g_pcts.values)]:
    adjusted = adjust_to_100(pcts, decimals=1)
    for bar, val in zip(bars, adjusted):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + y_max * 0.012,
                f'{val:.1f}%',
                ha='center', va='bottom', fontsize=8, fontweight='bold',
                color='#333333')

# draw rate gap annotation
draw_gap = hf_pcts[0] - g_pcts[0]
gap_y    = max(hf_pcts[0], g_pcts[0]) + y_max * 0.20
ax.annotate(
    f'Filtered vs GPT gap: {draw_gap:.1f}%',
    xy=(x[0] + width/2, gap_y),
    ha='center', fontsize=8.5, color='#c0392b', fontweight='bold'
)
ax.annotate(
    '', xy=(x[0] + width,     hf_pcts[0] + y_max * 0.09),
    xytext=(x[0],             hf_pcts[0] + y_max * 0.09),
    arrowprops=dict(arrowstyle='-', color='#c0392b', lw=1.2)
)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Share of votes (%)', fontsize=11)
ax.set_title(
    'Vote option distribution: human raw, human filtered and GPT-5.2 validated votes\n'
    'Filtering removes inattentive and automated voting patterns',
    fontsize=11, fontweight='bold', pad=12, loc='left'
)
ax.legend(fontsize=9, frameon=True, framealpha=0.9, edgecolor='#cccccc')
ax.spines[['top', 'right']].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.grid(axis='y', alpha=0.2, linestyle='--', color='#aaaaaa')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0f}%'))

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, 'vote_distribution_comparison.pdf')
plt.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ── DIAGNOSTIC: check exact vote counts and percentages ──────────────────────

print('=' * 60)
print('HUMAN RAW')
print('=' * 60)
hr_counts_full = df_human['vote'].value_counts().sort_index()
hr_total_full  = len(df_human)
for vote, count in hr_counts_full.items():
    pct = count / hr_total_full * 100
    print(f'  Vote {vote}: {count:,} votes | {pct:.6f}%')
print(f'  Total   : {hr_total_full:,}')
print(f'  Sum pct : {hr_counts_full.sum() / hr_total_full * 100:.6f}%')

print()
print('=' * 60)
print('HUMAN FILTERED')
print('=' * 60)
hf_counts_full = filtered_human_df['vote'].value_counts().sort_index()
hf_total_full  = len(filtered_human_df)
for vote, count in hf_counts_full.items():
    pct = count / hf_total_full * 100
    print(f'  Vote {vote}: {count:,} votes | {pct:.6f}%')
print(f'  Total   : {hf_total_full:,}')
print(f'  Sum pct : {hf_counts_full.sum() / hf_total_full * 100:.6f}%')

print()
print('=' * 60)
print('GPT 25k')
print('=' * 60)
g_counts_full = df_gpt_25k['vote'].value_counts().sort_index()
g_total_full  = len(df_gpt_25k)
for vote, count in g_counts_full.items():
    pct = count / g_total_full * 100
    print(f'  Vote {vote}: {count:,} votes | {pct:.6f}%')
print(f'  Total   : {g_total_full:,}')
print(f'  Sum pct : {g_counts_full.sum() / g_total_full * 100:.6f}%')

print()
print('=' * 60)
print('ROUNDED VALUES AT 2 DECIMAL PLACES')
print('=' * 60)
for label, counts, total in [
    ('Human raw',      hr_counts_full, hr_total_full),
    ('Human filtered', hf_counts_full, hf_total_full),
    ('GPT 25k',        g_counts_full,  g_total_full),
]:
    pcts   = [(c / total * 100) for c in counts.values]
    rounded = [round(p, 2) for p in pcts]
    print(f'  {label}: {rounded} -> sum = {sum(rounded):.2f}%')